In [1]:
import os
from bs4 import BeautifulSoup
import time
import urllib.request


In [2]:
SEASONS = [x for x in range(2016,2027)]
DATA_DIR = "data"
STANDINGS_DIR = os.path.join(DATA_DIR,"standings")
SCORES_DIR = os.path.join(DATA_DIR,"scores")

In [3]:
def get_html(url,selector, sleep_time = 5, retries=3):
    html = None
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    for i in range(1, retries+1):
        time.sleep(sleep_time*i)
        try:
            req = urllib.request.Request(url,headers=headers)
            with urllib.request.urlopen(req) as response:
                html = response.read().decode('utf-8')

                soup = BeautifulSoup(html, "html.parser")
                element = soup.select_one(selector)
                if element:
                    # Return the inner HTML of the selected element
                    print(f"Successfully scraped {url} with selector {selector}")
                    return str(element)
                else:
                    print(f"Selector '{selector}' not found on {url}")
                    return None
        
        except Exception as e:
            print(f"Attempt {i} failed for {url}: {e}")
            if i < retries:
                print(f"Retrying in {sleep_time*i} seconds...")
            else:
                print("Max retries reached")
                return None

In [4]:
def scrape_season(season):
    url = f"https://www.basketball-reference.com/leagues/NBA_{season}_games.html"
    html = get_html(url, "#content .filter")

    if not html:
        return
    soup = BeautifulSoup(html)
    links = soup.find_all("a")
    href = [l["href"] for l in links]
    standings_pages = [f"https://www.basketball-reference.com{l}" for l in href]
    
    for url in standings_pages:
        save_path = os.path.join(STANDINGS_DIR, url.split("/")[-1])
        # print(f"URL: {url}")
        # print(f"SAVE PATH: {save_path}")
        if os.path.exists(save_path):
            # !!TODO change it for present time months because matches exists but box scores are missing
            continue

        html = get_html(url, "#div_schedule")
        
        if html:
            with open(save_path, "w+", encoding="utf-8") as f:
                f.write(html)
        

In [ ]:
for season in SEASONS:
    scrape_season(season)

In [6]:
standings_files = os.listdir(STANDINGS_DIR)

In [7]:
def scrape_game(standings_file):
    with open(standings_file,'r') as f:
        html = f.read()

    soup = BeautifulSoup(html)
    links = soup.find_all("a")
    hrefs = [l.get("href") for l in links]
    box_scores = [l for l in hrefs if l and "boxscores" in l and ".html" in l]
    box_scores = [f"https://www.basketball-reference.com{l}" for l in box_scores]

    for url in box_scores:
        save_path = os.path.join(SCORES_DIR,url.split("/")[-1])
        if os.path.exists(save_path):
            continue

        html = get_html(url,"#content")
        if not html:
            continue

        with open(save_path,"w+",encoding="utf-8") as f:
            f.write(html)


In [8]:
for f in standings_files:
    filepath = os.path.join(STANDINGS_DIR,f)

    scrape_game(filepath)

Successfully scraped https://www.basketball-reference.com/boxscores/202503070LAC.html with selector #content
Successfully scraped https://www.basketball-reference.com/boxscores/202503100HOU.html with selector #content
Successfully scraped https://www.basketball-reference.com/boxscores/202503100MEM.html with selector #content
Successfully scraped https://www.basketball-reference.com/boxscores/202503100OKC.html with selector #content
Successfully scraped https://www.basketball-reference.com/boxscores/202503130GSW.html with selector #content
Successfully scraped https://www.basketball-reference.com/boxscores/202503140MIA.html with selector #content
Successfully scraped https://www.basketball-reference.com/boxscores/202503140PHI.html with selector #content
Successfully scraped https://www.basketball-reference.com/boxscores/202503140ATL.html with selector #content
Successfully scraped https://www.basketball-reference.com/boxscores/202503140HOU.html with selector #content
Successfully scrape